In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from gbmcollaboration import AllDataCenters
from gbmcollaboration import (
    SURVIVAL_EXCLUSION
)
from gbmcollaboration.utils import pathology_annotation_loader as patho
from gbmcollaboration.viz.cell2loc_plotting import clustermap, dotplot_composite_score
from gbmcollaboration.viz.stacked_barplot import clustered_stacked_barplot
from gbmcollaboration.viz.figure_helpers import savefig, set_mpl_rcparams
from gbmcollaboration.viz import set_mpl_rcparams, savefig
set_mpl_rcparams()
# from gbmcollaboration.utils import get_datasets, s3_ls
# data = get_datasets()

# Params

In [ ]:
from IPython.display import FileLink
FileLink('chunked_archive.tar.gz.aa')

In [ ]:
path_efs = '/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/'
path_ecm_annot = path_efs+'Analysis_Repo/Notebooks/ecm_neighborhood_annotations_v4.csv'
path_til_annot = path_efs+'results/visium/til_zones/til_zone_annotations.csv'
path_cell_type_info = '../../gbmcollaboration/metadata/Cell type annotation markers Glimmer.xlsx'
FIGURES_DIR = Path(path_efs+'figures/visium/cell2loc_colocalization/')
FIGURES_DIR_CUSTOM = Path(path_efs+'figures/visium/custom_colocalization/')
FIGURES_DIR_CUSTOM_NO_CLIP = Path(path_efs+'figures/visium/custom_colocalization_no_clip/')
FIGURES_DIR_CUSTOM_LOG2FC = Path(path_efs+'figures/visium/custom_colocalization_log2fc/')
FIGURES_DIR_CUSTOM_PATHO = Path(path_efs+'figures/visium/custom_colocalization_patho/')
FIGURES_DIR_PATHO = Path(path_efs+'figures/visium/colocalization_patho/')
FIGURES_DIR_BARPLOTS = Path(path_efs+'figures/visium/stacked_patient_barplots/')

# Load annotations

In [ ]:
# load ECM and TIL annotations
df_ecm_annot = pd.read_csv(path_ecm_annot,index_col='spot_id')
df_til_annot = pd.read_csv(path_til_annot,index_col='spot_id')

# load pathology annotations
cfg = patho.DataConfig(
    s3_bucket="pa-bi4jtrdsa151p4d3m-jepjxh9qrwgxphe8hd3r6exicahnoeuw1b-s3alias",
    s3_prefixes=(
        "fd319842-9157-4f04-bd4f-7a36ce6a0ba7/Non_mosaic_upload/Mosaic Annotations/",  # CHUV
        "2ca49430-8b5c-4504-ad96-344019c90e84/",  # UKER
    ),
)

acfg = patho.AnalysisConfig(
    min_spots_per_annotation=0,
    keep_only_shared_annotations=False,
)

df_patho_annot = patho.load_and_prepare(cfg, acfg)
df_all_annot = df_ecm_annot.join([df_til_annot[['TIL_positive','TIL_zone']], df_patho_annot['pathology_annotations']])
df_all_annot[['ECM_leiden','ECM_neighborhood']]=df_all_annot[['ECM_leiden','ECM_neighborhood']].astype(int)

# Load data

In [ ]:
adata = AllDataCenters.load_spatial_adata(
    exclude_low_quality_visium = True,
    exclude_idh1_mut = True,
    exclude_patient_ids = True,
    exclude_relapse = True,
    load_deconvolution=True,
)

df_q05 = adata.obs[adata.obs.columns.difference(['sample','imagerow','imagecol'])].fillna(0.)

# map cell types names to paper names and create simplified deconv matrix
df_markers, df_ct_map = pd.read_excel(path_cell_type_info,sheet_name=None).values()
assert all(df_q05.columns.isin(df_ct_map['celltype_level4_scanvi']))

# assert index matches with metadata
assert all(df_q05.index == df_all_annot.index)

# normalize
df_q05_norm = df_q05.div(df_q05.sum(axis=1), axis=0)

# join with metadata for each level and create simplified deconv matrix
# also get average per annotation
annot_keys = [
    'TIL_zone',
    'ECM_neighborhood',
    'pathology_annotations'
]

dfs_q05 = {}
dfs_q05_norm = {}
dfs_q05_sum_per_annot = {}
dfs_q05_norm_mean_per_annot = {}
for i in tqdm(range(1,5)):
    lvl = f'celltype_level{i}_curated'
    _map = df_ct_map.set_index('celltype_level4_scanvi')[lvl]

    dfs_q05[lvl] = df_q05.T.groupby(_map).sum().T.join(df_all_annot)
    dfs_q05_norm[lvl] = df_q05_norm.T.groupby(_map).sum().T.join(df_all_annot)

    ct_cols = dfs_q05_norm[lvl].columns.difference(df_all_annot.columns)

    dfs_q05_sum_per_annot[lvl] = {}
    dfs_q05_norm_mean_per_annot[lvl] = {}
    for k in annot_keys:
        dfs_q05_sum_per_annot[lvl][k] = dfs_q05[lvl].groupby(k)[ct_cols].sum(0).T
        dfs_q05_norm_mean_per_annot[lvl][k] = dfs_q05_norm[lvl].groupby(k)[ct_cols].mean(0).T

In [ ]:

df_q05 = adata.obs[adata.obs.columns.difference(['sample','imagerow','imagecol'])].fillna(0.)

# map cell types names to paper names and create simplified deconv matrix
df_markers, df_ct_map = pd.read_excel(path_cell_type_info,sheet_name=None).values()
assert all(df_q05.columns.isin(df_ct_map['celltype_level4_scanvi']))

# assert index matches with metadata
assert all(df_q05.index == df_all_annot.index)

# normalize
df_q05_norm = df_q05.div(df_q05.sum(axis=1), axis=0)

# join with metadata for each level and create simplified deconv matrix
# also get average per annotation
annot_keys = [
    'TIL_zone',
    'ECM_neighborhood',
    'pathology_annotations'
]

dfs_q05 = {}
dfs_q05_norm = {}
dfs_q05_sum_per_annot = {}
dfs_q05_norm_mean_per_annot = {}
for i in tqdm(range(1,5)):
    lvl = f'celltype_level{i}_curated'
    _map = df_ct_map.set_index('celltype_level4_scanvi')[lvl]

    dfs_q05[lvl] = df_q05.T.groupby(_map).sum().T.join(df_all_annot)
    dfs_q05_norm[lvl] = df_q05_norm.T.groupby(_map).sum().T.join(df_all_annot)

    ct_cols = dfs_q05_norm[lvl].columns.difference(df_all_annot.columns)

    dfs_q05_sum_per_annot[lvl] = {}
    dfs_q05_norm_mean_per_annot[lvl] = {}
    for k in annot_keys:
        dfs_q05_sum_per_annot[lvl][k] = dfs_q05[lvl].groupby(k)[ct_cols].sum(0).T
        dfs_q05_norm_mean_per_annot[lvl][k] = dfs_q05_norm[lvl].groupby(k)[ct_cols].mean(0).T

In [ ]:
df_gex = adata.to_df('log_norm_count')
df_gex_ecm_mean = df_gex.groupby(df_all_annot['ECM_neighborhood']).mean()
df_gex_ecm_mean.to_csv(path_efs+'supplementary_tables/4b_ECM_neighborhood_mean_gene_expression.csv')

for ct_lvl in dfs_q05_norm_mean_per_annot.keys():
    for k in dfs_q05_norm_mean_per_annot[ct_lvl].keys():
        dfs_q05_norm_mean_per_annot[ct_lvl][k].to_csv(path_efs+f'supplementary_tables/4c_{k}_{ct_lvl}_mean_cell_type_composition.csv')

# Cell2loc NMF

# Cell2loc dotplots

In [ ]:
plot_annots = {
    'TIL_zone': None,
    'ECM_neighborhood': None,
    'pathology_annotations': None
}

for i in tqdm(range(1,5)):
    lvl = f'celltype_level{i}_curated'

    for k, cat_subset in plot_annots.items():

        if k == 'ECM_neighborhood':
            col_prefix = 'ECM_'
        elif k == 'TIL_zone':
            col_prefix = ''

        df_q05_norm_mean = dfs_q05_norm_mean_per_annot[lvl][k]

        upper = np.quantile(df_q05_norm_mean,0.99)
        df_plot = df_q05_norm_mean.copy()
        df_plot = df_plot.clip(upper=upper)
        df_plot.columns = col_prefix+df_plot.columns.astype(str)

        if cat_subset is not None:
            df_plot = df_plot.loc[cat_subset]

        clustermap(df_plot, figure_size=(14, 14), 
                    cmap='RdPu', log=True, fun_type='dotplot',
                    cluster_rows=True, cluster_cols=True)


        # save plot
        out_path = Path(FIGURES_DIR) / f'dotplot_{lvl}_{k}.png'
        out_path.parent.mkdir(parents=True, exist_ok=True)
        savefig(df_plot, out_path, close=True)

# Custom dotplots

In [ ]:
plot_annots = {
    'TIL_zone': None,
    'ECM_neighborhood': None,
    'pathology_annotations': None
}

for i in tqdm(range(1,5)):
    lvl = f'celltype_level{i}_curated'

    for k, cat_subset in plot_annots.items():

        if k == 'ECM_neighborhood':
            col_prefix = 'ECM_'
        elif k == 'TIL_zone':
            col_prefix = ''

        df_q05_sum = dfs_q05_sum_per_annot[lvl][k]

        # upper = np.quantile(df_q05_sum,0.99)
        df_plot = df_q05_sum.copy()
        # df_plot = df_plot.clip(upper=upper)
        df_plot.columns = col_prefix+df_plot.columns.astype(str)

        if cat_subset is not None:
            df_plot = df_plot.loc[cat_subset]

        fig, ax, metrics = dotplot_composite_score(
            abund_per_nh=df_plot.T, 
            top_n_per_niche=5, 
            show_excluded_rows=False,
            center_zero=True,
            cmap='bwr'
        )

        # save plot
        # out_path = Path(FIGURES_DIR_CUSTOM) / f'dotplot_{lvl}_{k}.png'
        # out_path.parent.mkdir(parents=True, exist_ok=True)
        # savefig(df_plot, out_path, close=True)
        break
    break

# Custom dotplots no clipping

In [ ]:
plot_annots = {
    'TIL_zone': None,
    'ECM_neighborhood': None,
    'pathology_annotations': None
}

for i in tqdm(range(1,5)):
    lvl = f'celltype_level{i}_curated'

    for k, cat_subset in plot_annots.items():

        if k == 'ECM_neighborhood':
            col_prefix = 'ECM_'
        elif k == 'TIL_zone':
            col_prefix = ''

        df_q05_sum = dfs_q05_sum_per_annot[lvl][k]

        # upper = np.quantile(df_q05_sum,0.99)
        df_plot = df_q05_sum.copy()
        # df_plot = df_plot.clip(upper=upper)
        df_plot.columns = col_prefix+df_plot.columns.astype(str)

        if cat_subset is not None:
            df_plot = df_plot.loc[cat_subset]

        fig, ax, metrics = dotplot_composite_score(
            abund_per_nh=df_plot.T, 
            top_n_per_niche=5, 
            show_excluded_rows=False,
            center_zero=True,
            cmap='bwr'
        )

        # save plot
        out_path = Path(FIGURES_DIR_CUSTOM_NO_CLIP) / f'dotplot_{lvl}_{k}.png'
        out_path.parent.mkdir(parents=True, exist_ok=True)
        savefig(df_plot, out_path, close=True)

# Custom dotplots logfc only

In [ ]:
plot_annots = {
    'TIL_zone': None,
    'ECM_neighborhood': None,
    'pathology_annotations': None
}

for i in tqdm(range(1,5)):
    lvl = f'celltype_level{i}_curated'

    for k, cat_subset in plot_annots.items():

        if k == 'ECM_neighborhood':
            col_prefix = 'ECM_'
        elif k == 'TIL_zone':
            col_prefix = ''

        df_q05_sum = dfs_q05_sum_per_annot[lvl][k]

        # upper = np.quantile(df_q05_sum,0.99)
        df_plot = df_q05_sum.copy()
        # df_plot = df_plot.clip(upper=upper)
        df_plot.columns = col_prefix+df_plot.columns.astype(str)

        if cat_subset is not None:
            df_plot = df_plot.loc[cat_subset]

        fig, ax, metrics = plot_neighborhood_marker_dotplot_from_agg(
            abund_per_nh=df_plot.T, 
            top_n_per_niche=5, 
            show_excluded_rows=False,
            center_zero=True,
            marker_score_mode='log2fc',
            cmap='bwr'
        )

        # save plot
        out_path = Path(FIGURES_DIR_CUSTOM_LOG2FC) / f'dotplot_{lvl}_{k}.png'
        out_path.parent.mkdir(parents=True, exist_ok=True)
        savefig(df_plot, out_path, close=True)

# ECM vs pathology dotplots

In [ ]:
plot_annots = {
    'TIL_zone': None,
    'ECM_neighborhood': None,
}
for norm in ['row','col']:
    for k, cat_subset in plot_annots.items():

        if k == 'ECM_neighborhood':
            col_prefix = 'ECM_'
        elif k == 'TIL_zone':
            col_prefix = ''

        df_plot = pd.crosstab(df_all_annot['pathology_annotations'],df_all_annot[k])

        if norm == 'row':
            df_plot = df_plot.div(df_plot.sum(axis=1),axis=0)
        elif norm == 'col':
            df_plot = df_plot.div(df_plot.sum(axis=0),axis=1)

        upper = np.quantile(df_plot,0.99)
        df_plot = df_plot.clip(upper=upper)
        df_plot.columns = col_prefix+df_plot.columns.astype(str)

        if cat_subset is not None:
            df_plot = df_plot.loc[cat_subset]

        clustermap(df_plot, figure_size=(7, 5), 
                    cmap='RdPu', log=True, fun_type='dotplot',
                    cluster_rows=True, cluster_cols=True)

        # save plot
        out_path = Path(FIGURES_DIR_PATHO) / f'dotplot_{k}_{norm}.png'
        out_path.parent.mkdir(parents=True, exist_ok=True)
        savefig(df_plot, out_path, close=False)

# ECM vs pathology custom dotplots

In [ ]:
plot_annots = {
    'TIL_zone': None,
    'ECM_neighborhood': None,
}

for k, cat_subset in plot_annots.items():

    if k == 'ECM_neighborhood':
        col_prefix = 'ECM_'
    elif k == 'TIL_zone':
        col_prefix = ''

    df_plot = pd.crosstab(df_all_annot['pathology_annotations'],df_all_annot[k])

    df_plot.columns = col_prefix+df_plot.columns.astype(str)

    if cat_subset is not None:
        df_plot = df_plot.loc[cat_subset]

    fig, ax, metrics = dotplot_composite_score(
        abund_per_nh=df_plot.T, 
        top_n_per_niche=50, 
        show_excluded_rows=False,
        # size_floor=20,
        size_scale=200,
        clip_percentile=99,
        fig_w=7.5
    )

    # save plot
    out_path = Path(FIGURES_DIR_CUSTOM_PATHO) / f'dotplot_{k}.png'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    savefig(df_plot, out_path, close=False)

# Patient composition plots

In [ ]:
plot_annots = {
    'TIL_zone': None,
    'ECM_neighborhood': None,
    'pathology_annotations': None
}

for k, cat_subset in plot_annots.items():

    if k == 'ECM_neighborhood':
        col_prefix = 'ECM_'
    elif k == 'TIL_zone':
        col_prefix = ''

    df_patient_props = pd.crosstab(df_all_annot['sample'],df_all_annot[k])
    df_patient_props = df_patient_props.div(df_patient_props.sum(axis=1),axis=0)
    
    if k == 'TIL_zone':
        cat_order = df_patient_props.sum(0).sort_values(ascending=False).index.tolist()
        df_patient_props = df_patient_props.sort_values(cat_order)
        cluster_cols=False
        cluster_rows=False
    else:
        cluster_cols=True
        cluster_rows=True


    col_annot = df_patient_props.index.str.split('_').str[0]

    fig, ax, ordered = clustered_stacked_barplot(
        df_patient_props,
        cluster_cols=cluster_cols,
        cluster_rows=cluster_rows,
        col_annotation=col_annot,
        col_annotation_cmap="Set2",
        col_annotation_label="Center",
        figsize=(25,8),
        legend_kwargs = dict(
            fontsize=20,
            title_fontsize=20,
        )
    )
    # save plot
    out_path = Path(FIGURES_DIR_BARPLOTS) / f'barplot_{k}.png'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    savefig(df_patient_props, out_path, close=True)